# #214 causal decision-epoch market/SP coverage

This notebook is a bounded, read-only rendering of the immutable package. It does not train, rank, settle, or authorize bets.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('src').resolve()))
import json
import pandas as pd
from IPython.display import Markdown, display

package = Path('results/bj214_decision_epoch_tensor_v1')
decision = json.loads((package / 'analysis/decision.json').read_text())
coverage = pd.read_csv(package / 'audit/coverage_by_cohort_rule.csv')
model_gate = pd.read_csv(package / 'audit/representative_model_support_gate.csv')
ticket_gate = pd.read_csv(package / 'audit/ticket_operability_gate.csv')
display(Markdown(
    f"**Disposition:** `{decision['disposition']}`  \
"
    f"**Model ready for #204:** `{decision['model_ready_for_204']}`  \
"
    f"**Ticket operability:** `{decision['ticket_operability_disposition']}`  \
"
    f"**Controlling rule:** `{decision['representative_rule']['epoch_rule']}`\n\n"
    'Model readiness depends only on model-learning support and causal safety. '
    'Ticket operability gates only a later ticket/operational stage and does not gate #204.'
))
display(Markdown('## Model-learning support (controlling gate)'))
display(model_gate)
display(Markdown('## Ticket operability (separate gate)'))
display(ticket_gate)
sensitivity = decision['prospective_activation_sensitivity']
display(Markdown(
    '## Prospective activation sensitivity (non-controlling)\n'
    f"{sensitivity['causal_core_complete_matches']} / "
    f"{sensitivity['candidate_matches']} causal core matches "
    f"({sensitivity['model_core_coverage']:.2%})."
))
model_blockers = decision['blocking_facts']['model_learning']
ticket_blockers = decision['blocking_facts']['ticket_operability']
display(Markdown(
    '## Model-learning remediation alternatives\n- '
    + '\n- '.join(model_blockers)
))
display(Markdown(
    '## Ticket-operability boundary\n- '
    + '\n- '.join(ticket_blockers)
))
display(Markdown('## Full epoch-family sensitivity (non-controlling)'))
coverage

**Disposition:** `BLOCKED_INSUFFICIENT_CAUSAL_SUPPORT`  **Model ready for #204:** `False`  **Ticket operability:** `BLOCKED_TICKET_OPERABILITY`  **Controlling rule:** `MEDIAN_KICKOFF_MINUS_10H`

Model readiness depends only on model-learning support and causal safety. Ticket operability gates only a later ticket/operational stage and does not gate #204.

## Model-learning support (controlling gate)

,cohort,representative_epoch_rule,evaluated_issues,required_issues,evaluated_epochs,model_n7_passing_epochs,model_n7_epoch_pass_rate,required_epoch_pass_rate,candidate_matches,causal_core_complete_matches,model_core_coverage,required_model_core_coverage,cohort_model_gate_pass
0,development,MEDIAN_KICKOFF_MINUS_10H,342,30,342,0,0.000000,0.8,67651,0,0.000000,0.8,False
1,historical_model_unseen,MEDIAN_KICKOFF_MINUS_10H,52,5,52,12,0.230769,0.8,2087,1153,0.552468,0.8,False
2,prospective_post_activation,MEDIAN_KICKOFF_MINUS_10H,0,2,0,0,0.000000,0.8,0,0,0.000000,0.8,False


## Ticket operability (separate gate)

,cohort,representative_epoch_rule,evaluated_issues,required_issues,evaluated_epochs,ticket_n7_passing_epochs,ticket_n7_epoch_pass_rate,required_epoch_pass_rate,candidate_matches,strict_complete_eligible_matches,ticket_eligible_coverage,required_ticket_eligible_coverage,cohort_ticket_gate_pass
0,development,MEDIAN_KICKOFF_MINUS_10H,342,30,342,0,0,0.8,67651,0,0,0.8,False
1,historical_model_unseen,MEDIAN_KICKOFF_MINUS_10H,52,5,52,0,0,0.8,2087,0,0,0.8,False
2,prospective_post_activation,MEDIAN_KICKOFF_MINUS_10H,0,2,0,0,0,0.8,0,0,0,0.8,False


## Prospective activation sensitivity (non-controlling)
269 / 289 causal core matches (93.08%).

## Model-learning remediation alternatives
- Development has no timestamped causal EURO plus Beijing-SP core support at the representative epoch rule.
- Historical named 1h/10h fields lack observation timestamps and remain diagnostic-only.
- The prospective cohort is below the predeclared issue minimum at the representative rule.
- Remediation alternatives are: obtain timestamped historical development support; accumulate a sufficient prospective training panel; or formally narrow #204 to a simpler or prospective-only design.

## Ticket-operability boundary
- No source supplies verified p34 sale-open and parlay-allowed state at decision epochs.
- Sale/parlay recovery can repair ticket operability, but cannot by itself create a causal historical model-learning panel.

## Full epoch-family sensitivity (non-controlling)

,cohort,epoch_rule,issues,epochs,evaluated_epochs,candidate_matches,causal_core_complete_matches,strict_complete_eligible_matches,model_n7_passing_epochs,ticket_n7_passing_epochs,model_core_coverage,ticket_eligible_coverage,model_n7_epoch_pass_rate,ticket_n7_epoch_pass_rate,controlling_representative_rule
0,development,FIRST_KICKOFF_MINUS_10H,342,342,342,0,0,0,0,0,0.000000,0,0.000000,0,False
1,development,FIRST_KICKOFF_MINUS_1H,342,342,342,0,0,0,0,0,0.000000,0,0.000000,0,False
2,development,ISSUE_START,342,342,342,100601,0,0,0,0,0.000000,0,0.000000,0,False
3,development,LAST_KICKOFF_MINUS_10H,342,342,342,2107,0,0,0,0,0.000000,0,0.000000,0,False
4,development,LAST_KICKOFF_MINUS_1H,342,342,342,520,0,0,0,0,0.000000,0,0.000000,0,False
5,development,MEDIAN_KICKOFF_MINUS_10H,342,342,342,67651,0,0,0,0,0.000000,0,0.000000,0,True
6,development,MEDIAN_KICKOFF_MINUS_1H,342,342,342,56625,0,0,0,0,0.000000,0,0.000000,0,False
7,historical_model_unseen,FIRST_KICKOFF_MINUS_10H,52,52,52,128,116,0,2,0,0.906250,0,0.038462,0,False
8,historical_model_unseen,FIRST_KICKOFF_MINUS_1H,52,52,52,128,128,0,2,0,1.000000,0,0.038462,0,False
9,historical_model_unseen,ISSUE_START,52,52,52,2788,336,0,14,0,0.120516,0,0.269231,0,False


In [2]:
from decision_epoch_tensor.builder import verify_package
verify_package(package)

{'contract_version': 'bj214-decision-epoch-tensor-v1',
 'disposition': 'BLOCKED_INSUFFICIENT_CAUSAL_SUPPORT',
 'epochs': 2855,
 'strict_roster_rows': 0,
 'strict_cells': 0,
 'features': 24,
 'outcome_namespace': 'PHYSICALLY_ABSENT',
 'source_manifest_sha256': 'f05e02404a91bb63d2a40c55bbe572964cf681057df72f7c69c6145c4c78572d',
 'status': 'PASS'}